# Experiment 2d — Order ablation: NONE vs NONE_REV

Tests whether the exp1b instruction-commitment probe tracks **content** (which instruction the
model committed to) or **position** (whichever instruction came first in the merged user field).

Same 200 held-out evolved pairs as exp2b (`seed=42`). Two conditions:

- `NONE`     — user field = `f"{s}\n\n{u}"` (s-first, original)
- `NONE_REV` — user field = `f"{u}\n\n{s}"` (u-first, reversed)

Generates with the HF model + prefill hooks, judges via vLLM (no-think Qwen3.5),
then rescores at all probe positions and prints the per-condition correlation table.

Outcomes:
- **Same-sign r** in NONE and NONE_REV → probe is order-invariant (instruction-commitment).
- **Sign-flipped r** → probe tracks position, not content.
- **r near zero in NONE_REV** → partial position dependence.

In [ ]:
# Install
import os, sys, importlib, site
REPO_DIR = "/content/Mech_spoof"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/ChuloIva/Mech_spoof.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q
!pip install -q -e {REPO_DIR}
os.environ["MECH_SPOOF_ROOT"] = REPO_DIR
src_path = f"{REPO_DIR}/src"
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()
site.main()
import mech_spoof
print("mech_spoof loaded from:", mech_spoof.__file__)

In [ ]:
# Auth + Drive
import os
from google.colab import drive, userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
drive.mount("/content/drive")

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = "qwen"
DRIVE_ROOT = Path("/content/drive/MyDrive/mech_spoof_results")

slug = MODEL_CONFIGS[MODEL_KEY].slug
EXP1B_DIR = DRIVE_ROOT / slug / "exp1b_authority_conflict"
OUT_DIR   = DRIVE_ROOT / slug / "exp2d_order_ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

JUDGE_MODEL_ID = "Qwen/Qwen3.5-9B"   # same judge family as exp2b
N_PAIRS = 200

print("EXP1B_DIR =", EXP1B_DIR, " exists=", EXP1B_DIR.exists())
print("OUT_DIR   =", OUT_DIR)

In [ ]:
# Run exp2b with conditions=(NONE, NONE_REV) on the same 200 pairs (seed=42)
from mech_spoof.experiments.exp2b_conflict_evolved import run_experiment_2b

result = run_experiment_2b(
    MODEL_KEY,
    OUT_DIR,
    exp1_dir=EXP1B_DIR,
    probe_position="response_first",   # default best-position from exp2c
    judge_model_id=JUDGE_MODEL_ID,
    n_pairs=N_PAIRS,
    conditions=("NONE", "NONE_REV"),
    judge_enable_thinking=False,
    free_after=True,
)
result.summary

## Recovery: judge-only (run if cell above crashed before vLLM judging)

Use this when generation finished and `generations.jsonl` was written, but the vLLM judge
step crashed (e.g. vllm wasn't installed). It loads vLLM, judges the saved generations,
and writes `result.json` — same shape as a complete `run_experiment_2b` run, so the
rescore + correlation cells below work unchanged.

In [ ]:
# Install vLLM (skipped on the original cell-1)
!pip install -q vllm

In [ ]:
# Judge-only recovery: reads OUT_DIR/generations.jsonl, runs vLLM judge, writes result.json
from mech_spoof.experiments import judge_generations_only

result = judge_generations_only(
    out_dir=OUT_DIR,
    model_key=MODEL_KEY,
    exp1_dir=EXP1B_DIR,
    probe_position="response_first",
    judge_model_id=JUDGE_MODEL_ID,
    judge_enable_thinking=False,
)
print(f"n_judged={result.n_judged}/{result.n_pairs * 2}")
result.summary

## Rescore at all positions and compare per-condition r

In [ ]:
# Build judged_*.csv from result.json so rescore_positions can join verdicts
import json, csv
from pathlib import Path

RESULT_JSON = OUT_DIR / "result.json"
JUDGE_CSV   = OUT_DIR / "judged.csv"

with open(RESULT_JSON) as f:
    rj = json.load(f)

with open(JUDGE_CSV, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["pair_idx", "condition", "judge_verdict", "system_followed", "judge_reason"])
    for r in rj["rows"]:
        sf = r.get("system_followed")
        sf_str = "" if sf is None else ("true" if sf else "false")
        w.writerow([r.get("pair_idx"), r.get("condition"),
                    r.get("judge_verdict", ""), sf_str, r.get("judge_reason", "")])
print("wrote", JUDGE_CSV)

In [ ]:
# Rescore: extract activations at all 3 positions for [prompt+response] of every row
from mech_spoof.experiments import rescore_exp2b_at_all_positions

OUT_CSV = OUT_DIR / "rescored_all_positions.csv"

summary = rescore_exp2b_at_all_positions(
    model_key=MODEL_KEY,
    exp1_dir=EXP1B_DIR,
    exp2b_result_path=RESULT_JSON,
    out_csv_path=OUT_CSV,
    judge_csv_path=JUDGE_CSV,
    batch_size=4,
    max_length=4096,
    max_response_chars=4000,
)
summary

In [ ]:
# Per-condition x per-position correlation table at the best layer
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

df = pd.read_csv(OUT_CSV)
if "system_followed" in df.columns:
    s = df["system_followed"].astype(str).str.lower()
    df["y"] = s.where(s.isin(["true", "false"])).map({"true": 1, "false": 0})

POSITIONS = ("response_first", "response_mid", "response_last")
CONDITIONS = ("NONE", "NONE_REV")

rows = []
for cond in CONDITIONS:
    sub = df[df["condition"] == cond]
    for pos in POSITIONS:
        col = f"probe_score_{pos}_best_layer"
        if col not in sub:
            continue
        x = pd.to_numeric(sub[col], errors="coerce").to_numpy()
        y = sub["y"].to_numpy(dtype=float)
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 3 or len(np.unique(y[m])) < 2:
            rows.append({"condition": cond, "position": pos, "n": int(m.sum()), "r": np.nan, "p": np.nan})
            continue
        r, p = pearsonr(x[m], y[m])
        rows.append({"condition": cond, "position": pos, "n": int(m.sum()), "r": float(r), "p": float(p)})

tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))
tbl.to_csv(OUT_DIR / "order_ablation_summary.csv", index=False)

In [ ]:
# Per-layer correlation curve, NONE vs NONE_REV side by side
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

POSITION_FOCUS = "response_first"
col = f"probe_scores_{POSITION_FOCUS}_by_layer_json"

fig, ax = plt.subplots(figsize=(11, 5))
for cond in ("NONE", "NONE_REV"):
    sub = df[df["condition"] == cond]
    layer_to = {}
    for _, r in sub.iterrows():
        if not isinstance(r[col], str) or not r[col].strip():
            continue
        sbl = json.loads(r[col])
        for L, sc in sbl.items():
            layer_to.setdefault(int(L), []).append((float(sc), r["y"]))
    layers, rs = [], []
    for L in sorted(layer_to):
        arr = [(x, yv) for x, yv in layer_to[L] if yv == yv]
        if len(arr) < 3:
            continue
        xs, ys = zip(*arr)
        if len(set(ys)) < 2:
            continue
        rv, _ = pearsonr(xs, ys)
        layers.append(L); rs.append(rv)
    ax.plot(layers, rs, marker=".", label=cond)

ax.axhline(0, color="gray", ls="--", lw=0.8)
ax.set_xlabel("Layer"); ax.set_ylabel(f"Pearson r at {POSITION_FOCUS}")
ax.set_title(f"{MODEL_KEY}: NONE vs NONE_REV per-layer correlation @ {POSITION_FOCUS}")
ax.legend(); plt.tight_layout()
plt.savefig(OUT_DIR / f"per_layer_corr_{POSITION_FOCUS}.png", dpi=120)
plt.show()